In [2]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# Transpose ข้อมูล
df = pd.read_excel("/content/Book5.xlsx")
df_T = df.T
df_T.columns = df_T.iloc[0]
df_T = df_T.iloc[1:].reset_index(drop=True)

# กรองเฉพาะตัวแปรที่เป็นตัวเลข
df_numeric = df_T.drop(columns=["!Sample_characteristics_ch1"], errors="ignore")
df_numeric = df_numeric.apply(pd.to_numeric, errors="coerce").fillna(0)

# กรองเฉพาะตัวแปรค่า CV อยู่ในช่วง [0.25,0.5]
def filter_columns_by_cv_only(df_numeric, cv_range=(0.25, 0.5)):
    mean_vals = df_numeric.mean(axis=0)
    std_vals = df_numeric.std(axis=0, ddof=1)
    cv_vals = std_vals / mean_vals.replace(0, np.nan)
    cond_cv = (cv_vals >= cv_range[0]) & (cv_vals <= cv_range[1])
    kept_cols = df_numeric.columns[cond_cv]
    return kept_cols

kept_cols = filter_columns_by_cv_only(df_numeric)
print(f"จำนวนตัวแปรที่เหลือหลังการกรอง CV (0.25–0.5): {len(kept_cols)}")
cols_to_keep = ["!Sample_characteristics_ch1"] + list(kept_cols)
df_T = df_T[cols_to_keep]

# แบ่งกลุ่มข้อมูล
def clean_and_select(df_T, keywords):
    df_sub = df_T[df_T["!Sample_characteristics_ch1"].isin(keywords)].copy()
    df_sub = df_sub.drop(columns=["!Sample_characteristics_ch1"])
    df_sub = df_sub.apply(pd.to_numeric, errors="coerce").fillna(0)
    return df_sub
df_AT  = clean_and_select(df_T, [
    "Histopathological diagnostic: astrocytoma, grade 3",
    "Histopathological diagnostic: astrocytoma, grade 2"
])
df_GBT = clean_and_select(df_T, [
    "Histopathological diagnostic: glioblastoma, grade 4",
    "Histopathological diagnostic:"
])
df_ODG = clean_and_select(df_T, [
    "Histopathological diagnostic: oligodendroglioma, grade 2",
    "Histopathological diagnostic: oligodendroglioma, grade 3"
])
df_NT  = clean_and_select(df_T, [
    "Histopathological diagnostic: non-tumor"
])

# ตรวจสอบขนาดของแต่ละกลุ่ม
print("\nShapes หลังการกรอง:")
for name, df_ in zip(["AT","GBT","ODG","NT"], [df_AT, df_GBT, df_ODG, df_NT]):
    print(name, df_.shape)

# แปลงเป็น NumPy
data_AT, data_GBT, data_ODG, data_NT = map(lambda x: x.to_numpy(), [df_AT, df_GBT, df_ODG, df_NT])
data_groups = [data_AT, data_GBT, data_ODG, data_NT]

# คำนวณสถิติทดสอบ T3
EPS = 1e-12
def _Bi_within(X):
    n = X.shape[0]
    if n < 2:
        return 0.0
    G = X @ X.T
    np.fill_diagonal(G, 0.0)
    return np.sum(G * G) / (n * (n - 1))

def _Bij_cross(X, Y):
    H = X @ Y.T
    return np.sum(H * H) / (X.shape[0] * Y.shape[0])

def ahmad_Tg_test(data_groups, alpha=0.05):
    g = len(data_groups)
    n_i = [grp.shape[0] for grp in data_groups]
    p = data_groups[0].shape[1]

    B_within = [_Bi_within(grp) for grp in data_groups]
    B_pairs = {}
    for i in range(g):
        for j in range(i + 1, g):
            B_pairs[(i, j)] = _Bij_cross(data_groups[i], data_groups[j])

    sum_Bi  = np.sum(B_within)
    sum_Bij = np.sum(list(B_pairs.values()))
    Tg = ((g - 1) * sum_Bi - 2.0 * sum_Bij) / (p * p)

    numerator_C3, P_star = 0.0, 0.0
    for i in range(g):
        for j in range(g):
            if i == j:
                continue
            ni, nj = n_i[i], n_i[j]
            key = (i, j) if i < j else (j, i)
            Bij = B_pairs[key]
            numerator_C3 += ni * nj * Bij
            P_star += ni * nj
    C3 = numerator_C3 / max(P_star, EPS)

    term1 = (g - 1)**2 * np.sum([1.0 / (ni ** 2) for ni in n_i])
    term2 = np.sum([2.0 / (n_i[i] * n_i[j]) for i in range(g) for j in range(i + 1, g)])
    A = term1 + term2
    sigma_hat = 2.0 * np.sqrt(max(A, 0.0)) * (C3 / (p * p))
    sigma_hat = max(sigma_hat, EPS)
    stat = Tg / sigma_hat
    p_value = 1 - norm.cdf(stat)
    return stat, p_value, Tg, sigma_hat

# เรียกใช้ฟังก์ชันของสถิติทดสอบ T3
stat, p_value, Tg, sigma_Tg0 = ahmad_Tg_test(data_groups)
print("\n===== ผลการทดสอบ =====")
print(f"Statistic Test (T3) = {stat:.6f}")
print(f"p-value             = {p_value:.6f}")

จำนวนตัวแปรที่เหลือหลังการกรอง CV (0.25–0.5): 24785

Shapes หลังการกรอง:
AT (26, 24785)
GBT (81, 24785)
ODG (50, 24785)
NT (23, 24785)

===== ผลการทดสอบ =====
Statistic Test (T3) = 2.046850
p-value             = 0.020336
